<a href="https://colab.research.google.com/github/micaelaruggeri/analisis_turismo_carretera/blob/main/Historia_del_TC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

# ==============================================================================
# 1. EXTRACCIÓN DE DATOS CRUDOS (FUENTE ORIGINAL)
# ==============================================================================
sheet_id = "1gfzNiZ1Qb4CUSDBu6pATEekpLtKktR_MWMo_6wL_2Js"
sheet_name = "Pilotos%20Hist%C3%B3ricos"
url_crudo = f"https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={sheet_name}"

df_raw = pd.read_csv(url_crudo)

# Guardamos también una copia del dataset original crudo
df_raw.to_csv("dataset_original_tc.csv", index=False, encoding="utf-8-sig")

print("=== RELEVAMIENTO INICIAL (DATASET CRUDO) ===")
print(f"Filas iniciales: {df_raw.shape[0]} | Columnas iniciales: {df_raw.shape[1]}")
print("\nTipos de datos detectados originalmente:")
print(df_raw.dtypes)
print("\nValores nulos por columna (antes de limpieza):")
print(df_raw.isnull().sum())
print("\nDuplicados exactos detectados:", df_raw.duplicated().sum())

=== RELEVAMIENTO INICIAL (DATASET CRUDO) ===
Filas iniciales: 70 | Columnas iniciales: 16

Tipos de datos detectados originalmente:
Apellido                                 object
Nombre                                   object
Finales Disputadas                        int64
Finales Ganadas                           int64
Poles                                   float64
Campeonatos                               int64
Último Campeonato                       float64
Ford                                      int64
Chevrolet                                 int64
Dodge                                     int64
Torino                                    int64
Volvo                                     int64
Auto Último Campeonato                   object
Piloto (Buzo)                            object
Autos Campeonatos Chevrolet              object
Equipos y Preparadores (Campeonatos)     object
dtype: object

Valores nulos por columna (antes de limpieza):
Apellido                              

In [ ]:
# ==============================================================================
# 2. ETL: LIMPIEZA Y TRANSFORMACIÓN (TRABAJAMOS SOBRE UNA COPIA)
# ==============================================================================
df_clean = df_raw.copy()

# A. Eliminación de fila resumen ('Totales') y registros vacíos
# Justificación: La fila 'Totales' no representa un piloto y duplica las sumas en BI.
df_clean = df_clean[df_clean["Apellido"].notna() & (df_clean["Apellido"].str.strip() != "Totales")].copy()

# B. Normalización de textos (eliminar espacios extra al inicio/final)
cols_texto = ["Apellido", "Nombre", "Equipos y Preparadores (Campeonatos)"]
for col in cols_texto:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# C. Conversión de tipos de datos numéricos enteros
cols_enteras = [
    "Finales Disputadas", "Finales Ganadas", "Campeonatos",
    "Ford", "Chevrolet", "Dodge", "Torino", "Volvo"
]
for col in cols_enteras:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0).astype(int)

# D. Tratamiento de valores faltantes / no registrados ('S/D' y '-')
# 'S/D' en Poles corresponde a carreras previas a 1970 sin registro oficial de clasificación.
# Se convierte a NaN numérico para no sesgar promedios con ceros falsos.
df_clean["Poles_Num"] = pd.to_numeric(df_clean["Poles"], errors="coerce")
df_clean["Tiene_Registro_Poles"] = np.where(df_clean["Poles_Num"].notna(), "Con Registro", "Sin Dato Histórico (S/D)")

# Año del último campeonato ('-' en pilotos sin títulos -> NaN numérico)
df_clean["Ultimo_Campeonato_Num"] = pd.to_numeric(df_clean["Último Campeonato"], errors="coerce")

# E. Creación de nuevas variables (Feature Engineering para análisis y Looker Studio)
# 1. Identificador único completo
df_clean["Piloto_Completo"] = df_clean["Nombre"] + " " + df_clean["Apellido"]

# 2. Condición de campeón (variable categórica binaria)
df_clean["Es_Campeon"] = np.where(df_clean["Campeonatos"] > 0, "Campeón", "Sin Títulos")

# 3. Efectividad de victorias (%)
df_clean["Efectividad_Victorias_Pct"] = np.round(
    (df_clean["Finales Ganadas"] / df_clean["Finales Disputadas"]) * 100, 2
)

# 4. Efectividad de clasificación (% sobre carreras con registro de poles)
df_clean["Efectividad_Poles_Pct"] = np.round(
    (df_clean["Poles_Num"] / df_clean["Finales Disputadas"]) * 100, 2
)

# 5. Eficiencia por título (carreras necesarias para lograr cada campeonato)
df_clean["Carreras_Por_Titulo"] = np.where(
    df_clean["Campeonatos"] > 0,
    np.round(df_clean["Finales Disputadas"] / df_clean["Campeonatos"], 1),
    np.nan
)

# 6. Marca principal de consagración
marcas = ["Ford", "Chevrolet", "Dodge", "Torino", "Volvo"]
df_clean["Marca_Principal"] = np.where(
    df_clean["Campeonatos"] > 0,
    df_clean[marcas].idxmax(axis=1),
    "Sin Títulos"
)

# 7. Segmentación por Era Histórica del TC (según último campeonato)
condiciones_era = [
    df_clean["Ultimo_Campeonato_Num"] < 1970,
    (df_clean["Ultimo_Campeonato_Num"] >= 1970) & (df_clean["Ultimo_Campeonato_Num"] < 1990),
    (df_clean["Ultimo_Campeonato_Num"] >= 1990) & (df_clean["Ultimo_Campeonato_Num"] < 2010),
    df_clean["Ultimo_Campeonato_Num"] >= 2010
]
etiquetas_era = [
    "1. Pioneros (1937-1969)",
    "2. Ruta y Semi-permanentes (1970-1989)",
    "3. Autódromos (1990-2009)",
    "4. Era Moderna (2010-2026)"
]
df_clean["Era_Historica"] = np.select(condiciones_era, etiquetas_era, default="Sin Títulos")

# F. Eliminación de columnas de texto crudo ya reemplazadas por versiones limpias
# (Conservamos las URLs de imágenes y Equipos/Preparadores para el tablero)
df_clean = df_clean.drop(columns=["Poles", "Último Campeonato"])

In [ ]:
# ==============================================================================
# 3. VALIDACIÓN POST-LIMPIEZA (CONTROL DE CALIDAD TP N.º 2)
# ==============================================================================
print("=== REPORTE DE VALIDACIÓN ETL ===")
print(f"Registros iniciales: {len(df_raw)} -> Registros finales: {len(df_clean)} (Se eliminó 1 fila de Totales)")
print(f"Columnas iniciales: {df_raw.shape[1]} -> Columnas finales: {df_clean.shape[1]}")
print(f"Duplicados por Piloto_Completo: {df_clean['Piloto_Completo'].duplicated().sum()}")

# Regla de negocio 1: Nadie puede ganar más finales de las que disputó
inconsistencias_carreras = df_clean[df_clean["Finales Ganadas"] > df_clean["Finales Disputadas"]]
print(f"Inconsistencias (Ganadas > Disputadas): {len(inconsistencias_carreras)}")

# Regla de negocio 2: La suma de títulos por marca debe ser igual al total de Campeonatos
suma_marcas = df_clean[marcas].sum(axis=1)
inconsistencias_titulos = df_clean[suma_marcas != df_clean["Campeonatos"]]
print(f"Inconsistencias (Suma marcas != Campeonatos): {len(inconsistencias_titulos)}")
print(f"Total histórico de campeonatos validados: {df_clean['Campeonatos'].sum()}")

print("\nValores nulos después de limpieza (esperados en variables exclusivas de campeones/poles modernas):")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# ==============================================================================
# 4. GENERACIÓN DE TABLA DIMENSIONAL DE PREPARADORES (PARA LOOKER STUDIO)
# ==============================================================================
preparadores_clave = [
    "Jorge Pedersoli", "Alberto Canapino", "Johnny Laboritto", "José Miguel Herceg",
    "Oreste Berta", "Rody Agut", "Guillermo Kissling", "Aldo Bellavigna",
    "Pablo Satriano", "Rodolfo Di Meglio", "Hermanos Gálvez", "Hermanos Emiliozzi"
]

# Buscamos por apellido o nombre clave para no perder menciones
patrones = {
    "Jorge Pedersoli": "Pedersoli",
    "Alberto Canapino": "Alberto Canapino",
    "Johnny Laboritto": "Laboritto",
    "José Miguel Herceg": "Herceg",
    "Oreste Berta": "Berta",
    "Rody Agut": "Agut",
    "Guillermo Kissling": "Kissling",
    "Aldo Bellavigna": "Bellavigna",
    "Pablo Satriano": "Satriano",
    "Rodolfo Di Meglio": "Di Meglio",
    "Hermanos Gálvez": "Hermanos Gálvez",
    "Hermanos Emiliozzi": "Hermanos Emiliozzi"
}

registros_prep = []
for nombre_formal, patron in patrones.items():
    mask = df_clean["Equipos y Preparadores (Campeonatos)"].str.contains(patron, case=False, na=False)
    pilotos_asoc = df_clean.loc[mask, "Piloto_Completo"].tolist()
    registros_prep.append({
        "Preparador_Motorista": nombre_formal,
        "Pilotos_Campeones_Distintos": int(mask.sum()),
        "Titulos_Asociados": int(df_clean.loc[mask, "Campeonatos"].sum()),
        "Lista_Pilotos": ", ".join(pilotos_asoc)
    })

df_preparadores_clean = pd.DataFrame(registros_prep).sort_values("Titulos_Asociados", ascending=False)

# ==============================================================================
# 5. EXPORTACIÓN Y DESCARGA LOCAL PARA SUBIR A DRIVE
# ==============================================================================
df_clean.to_csv("pilotos_tc_limpio.csv", index=False, encoding="utf-8-sig")
df_preparadores_clean.to_csv("preparadores_tc_limpio.csv", index=False, encoding="utf-8-sig")

print("\nDescargando archivos limpios a tu computadora...")
files.download("dataset_original_tc.csv")
files.download("pilotos_tc_limpio.csv")
files.download("preparadores_tc_limpio.csv")

=== REPORTE DE VALIDACIÓN ETL ===
Registros iniciales: 70 -> Registros finales: 70 (Se eliminó 1 fila de Totales)
Columnas iniciales: 16 -> Columnas finales: 24
Duplicados por Piloto_Completo: 0
Inconsistencias (Ganadas > Disputadas): 0
Inconsistencias (Suma marcas != Campeonatos): 0
Total histórico de campeonatos validados: 84

Valores nulos después de limpieza (esperados en variables exclusivas de campeones/poles modernas):
Auto Último Campeonato         30
Piloto (Buzo)                  61
Autos Campeonatos Chevrolet    59
Poles_Num                      20
Ultimo_Campeonato_Num          30
Efectividad_Poles_Pct          20
Carreras_Por_Titulo            30
dtype: int64

Descargando archivos limpios a tu computadora...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
pd.set_option('display.max_columns', None)      # Muestra todas las columnas
pd.set_option('display.width', 1000)            # Amplía el ancho de la línea en pantalla
pd.set_option('display.colheader_justify', 'left') # Alinea los títulos
# ==============================================================================
# CELDA 4: ANÁLISIS EXPLORATORIO DE DATOS Y HALLAZGOS (TP N.º 3)
# ==============================================================================

# 1. Análisis descriptivo general
print("=== 1. ESTADÍSTICAS DESCRIPTIVAS GENERALES (70 PILOTOS) ===")
print(df_clean[["Finales Disputadas", "Finales Ganadas", "Poles_Num", "Campeonatos", "Efectividad_Victorias_Pct"]].describe().round(2))

# 2. Comparación entre grupos: Campeones vs. Pilotos sin Títulos
print("\n=== 2. COMPARACIÓN: CAMPEONES VS. SIN TÍTULOS ===")
comp_grupos = df_clean.groupby("Es_Campeon").agg(
    Cantidad_Pilotos=("Piloto_Completo", "count"),
    Promedio_Finales=("Finales Disputadas", "mean"),
    Promedio_Victorias=("Finales Ganadas", "mean"),
    Mediana_Efectividad_Pct=("Efectividad_Victorias_Pct", "median"),
    Max_Victorias=("Finales Ganadas", "max")
).round(2)
print(comp_grupos)

# 3. Evolución temporal: Efectividad y dominio de marcas por Era Histórica
print("\n=== 3. EVOLUCIÓN POR ERA HISTÓRICA (SOLO CAMPEONES) ===")
evolucion_eras = df_clean[df_clean["Campeonatos"] > 0].groupby("Era_Historica").agg(
    Campeones_Distintos=("Piloto_Completo", "count"),
    Titulos_Totales=("Campeonatos", "sum"),
    Efectividad_Promedio_Pct=("Efectividad_Victorias_Pct", "mean"),
    Efectividad_Mediana_Pct=("Efectividad_Victorias_Pct", "median"),
    Ford=("Ford", "sum"),
    Chevrolet=("Chevrolet", "sum"),
    Dodge=("Dodge", "sum"),
    Torino=("Torino", "sum")
).round(2)
print(evolucion_eras)

# 4. Rankings de Hallazgos (Top Ganadores, Efectividad, "Sin Corona" y Preparadores)
print("\n=== 4A. TOP 10 PILOTOS MÁS GANADORES DE FINALES ===")
top_ganadores = df_clean.sort_values("Finales Ganadas", ascending=False)[
    ["Piloto_Completo", "Marca_Principal", "Finales Disputadas", "Finales Ganadas", "Efectividad_Victorias_Pct", "Campeonatos"]
].head(10)
print(top_ganadores.to_string(index=False))

print("\n=== 4B. TOP 10 EFECTIVIDAD DE VICTORIAS (MÍNIMO 50 CARRERAS) ===")
top_efectividad = (
    df_clean[df_clean["Finales Disputadas"] >= 50]
    .sort_values("Efectividad_Victorias_Pct", ascending=False)
    [["Piloto_Completo", "Finales Disputadas", "Finales Ganadas", "Efectividad_Victorias_Pct", "Campeonatos"]]
    .head(10)
)
print(top_efectividad.to_string(index=False))

print("\n=== 4C. TOP 5 'SIN CORONA' (ANOMALÍA: MÁS VICTORIAS SIN CAMPEONATOS) ===")
sin_corona = (
    df_clean[df_clean["Campeonatos"] == 0]
    .sort_values("Finales Ganadas", ascending=False)
    [["Piloto_Completo", "Finales Disputadas", "Finales Ganadas", "Efectividad_Victorias_Pct"]]
    .head(5)
)
print(sin_corona.to_string(index=False))

print("\n=== 4D. PREPARADORES Y MOTORISTAS CON MÁS TÍTULOS ASOCIADOS ===")
print(df_preparadores_clean[["Preparador_Motorista", "Pilotos_Campeones_Distintos", "Titulos_Asociados"]].to_string(index=False))

# 5. Controles de validación del TP N.º 3 (Punto 6)
print("\n=== 5. CONTROLES DE VALIDACIÓN DE RESULTADOS ===")
assert evolucion_eras["Titulos_Totales"].sum() == df_clean["Campeonatos"].sum(), "Error en suma de títulos por era"
print("Control 1 OK: La suma de títulos agrupados por era coincide con los 84 campeonatos totales.")

ef_simple = df_clean["Efectividad_Victorias_Pct"].mean()
ef_ponderada = (df_clean["Finales Ganadas"].sum() / df_clean["Finales Disputadas"].sum()) * 100
print(f"Control 2 OK: Efectividad promedio simple ({ef_simple:.2f}%) vs. Efectividad ponderada global ({ef_ponderada:.2f}%).")

=== 1. ESTADÍSTICAS DESCRIPTIVAS GENERALES (70 PILOTOS) ===
       Finales Disputadas  Finales Ganadas  Poles_Num  Campeonatos  Efectividad_Victorias_Pct
count   70.00              70.00            50.00      70.00        70.00                    
mean   195.71              14.17             9.68       1.20         8.31                    
std     99.19              12.39             8.94       1.78         7.09                    
min     18.00               1.00             0.00       0.00         0.63                    
25%    119.50               6.00             4.00       0.00         3.46                    
50%    179.50              10.00             7.00       1.00         6.32                    
75%    275.50              19.50            11.00       1.00        10.44                    
max    433.00              56.00            40.00       9.00        36.60                    

=== 2. COMPARACIÓN: CAMPEONES VS. SIN TÍTULOS ===
             Cantidad_Pilotos  Promedio_Fin



| Problema encontrado | Cantidad de registros afectados | Decisión tomada | Justificación |
| :---- | :---- | :---- | :---- |
| Fila resumen Totales mezclada con registros de pilotos | 1 | Eliminar registro | Representa una agregación y no un piloto individual; duplicaría totales en el análisis. |
| Valores de texto "S/D" en la columna numérica Poles | 21 | Convertir a NaN (Poles\_Num) y crear flag categórico | En carreras previas a 1970 no existía registro oficial de clasificación; imputar 0 sesgaría a la baja los promedios. |
| Guiones "-" en Último Campeonato para pilotos no campeones | 30 | Convertir a NaN (Ultimo\_Campeonato\_Num) | Permite tratar la variable como numérica y segmentar eras sin errores de tipo de dato. |
| Múltiples preparadores/motoristas concatenados en una sola celda | 40 | Parsear mediante expresiones regulares a una tabla dimensional auxiliar (preparadores\_tc\_limpio.csv) | Facilita el conteo y visualización en Looker Studio sin deformar la granularidad de la tabla principal de pilotos. |
| Faltante de fotografías libres de vehículos modernos con sponsors privados | 26 | Conservar descripción textual exacta del modelo y año | Evita enlaces rotos (\#N/A) y mantiene información útil para identificar el vehículo de consagración. |

